## Modelo de Detec??o de Modelos de Detec??o de Smishing

Ankit Kumar Jain, B.B. Gupta,
Rule-Based Framework for Detection of Smishing Messages in Mobile Environment,
Procedia Computer Science,
Volume 125,
2018,
Pages 617-623,
ISSN 1877-0509,
https://doi.org/10.1016/j.procs.2017.12.079.
(https://www.sciencedirect.com/science/article/pii/S1877050917328478)
Abstract: Smishing is a cyber-security attack, which utilizes Short Message Service (SMS) to steal personal credentials of mobile users. The trust level of users on their smart devices has attracted attackers for performing various mobile security attacks like Smishing. In this paper, we implement the rule-based data mining classification approach in the detection of smishing messages. The proposed approach identified nine rules which can efficiently filter smishing SMS from the genuine one. Further, our approach applies rule-based classification algorithms to train these outstanding rules. Since the SMS text messages are very short and generally written in Lingo language, we have used text normalization to convert them into standard form to obtain better rules. The performance of the proposed approach is evaluated, and it achieved more than 99% true negative rate. Furthermore, the proposed approach is very efficient for the detection of the zero hour attack too.
Keywords: Smishing; Mobile Phishing; Data mining; Short messaging service; Machine learning

> Reprodu??o de resultados

In [ ]:
import pandas
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
import ruleset

#### 1. Pr? processamento dos dados

In [ ]:
# Import the CSV dataset as a dataframe
# Since pandas is already imported in cell 1, we can use it directly
df = pandas.read_csv('SMSSpamCollectionDataset.csv', encoding='latin-1')
df = df[['label', 'text']]

# Display the first few rows to get a glimpse of the data

df

In [ ]:
# Download dos recursos necess?rios do NLTK
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True) 
nltk.download('stopwords', quiet=True)

def normalize_text(original_text) -> str:
    '''
    Recebe um SMS
    
    Retorna texto original normalizado (mais conservador para melhor performance)
    '''

    # Convert to lowercase
    text = original_text.lower()

    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()
    
    # Tokenize (m?todo mais simples se houver problemas com punkt)
    try:
        words = nltk.word_tokenize(text)
    except:
        # Fallback para tokeniza??o simples
        words = text.split()
    
    # Remove stopwords apenas as mais comuns (mais conservador)
    # IMPORTANTE: Preservar s?mbolos financeiros e matem?ticos mesmo que sejam curtos
    common_stopwords = {'the', 'a', 'an', 'and', 'or', 'but', 'in', 'on', 'at', 'to', 'for', 'of', 'with', 'by', 'is', 'are', 'was', 'were', 'be', 'been', 'have', 'has', 'had', 'do', 'does', 'did', 'will', 'would', 'could', 'should'}
    
    # S?mbolos financeiros e matem?ticos importantes para detec??o de spam/smishing
    important_symbols = {
        # S?mbolos financeiros
        '$', '?', '?', '?', '?', '?', 
        # S?mbolos matem?ticos
        '+', '-', '*', '/', '=', '<', '>', '?', '?', '?', '?', '?', '?',
        # Outros s?mbolos importantes  
        '%', '#', '@', '&', '!', '?'
    }
    
    # Manter palavra se: n?o ? stopword E (tem mais de 1 char OU ? s?mbolo importante)
    words = [word for word in words if word not in common_stopwords and (len(word) > 1 or word in important_symbols)]
    
    # N?O aplicar stemming agressivo - manter palavras mais ?ntegras
    # Preservar s?mbolos importantes e palavras relevantes
    words = [word for word in words if len(word) > 2 or word in important_symbols]
    
    # Join words back into a string
    normalized_text = ' '.join(words)
    
    return normalized_text

# Testar com um exemplo
sample_text = "Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005!"
print(f"Original: {sample_text}")
print(f"Normalizado: {normalize_text(sample_text)}")

(debugando o por qu? do s?mbolo de dolar estar sumindo e verificando que ainda funciona...)

In [ ]:
# DEBUG: Vamos rastrear passo a passo onde o $ est? sendo perdido
def debug_normalize_text(original_text):
    print(f"1. Original: '{original_text}'")
    
    # Convert to lowercase
    text = original_text.lower()
    print(f"2. Lowercase: '{text}'")
    
    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()
    print(f"3. Remove spaces: '{text}'")
    
    # Tokenize
    try:
        words = nltk.word_tokenize(text)
        print(f"4. NLTK tokenize: {words}")
    except:
        words = text.split()
        print(f"4. Simple split: {words}")
    
    # Remove stopwords
    common_stopwords = {'the', 'a', 'an', 'and', 'or', 'but', 'in', 'on', 'at', 'to', 'for', 'of', 'with', 'by', 'is', 'are', 'was', 'were', 'be', 'been', 'have', 'has', 'had', 'do', 'does', 'did', 'will', 'would', 'could', 'should'}
    important_symbols = {
        # S?mbolos financeiros
        '$', '?', '?', '?', '?', '?', 
        # S?mbolos matem?ticos
        '+', '-', '*', '/', '=', '<', '>', '?', '?', '?', '?', '?', '?',
        # Outros s?mbolos importantes  
        '%', '#', '@', '&', '!', '?'
    }
    
    # Manter palavra se: n?o ? stopword E (tem mais de 1 char OU ? s?mbolo importante)
    words_after_stopwords = [word for word in words if word not in common_stopwords and (len(word) > 1 or word in important_symbols)]
    print(f"5. After stopwords removal: {words_after_stopwords}")
    
    # Remove short words mas preserva s?mbolos importantes
    words_after_length = [word for word in words_after_stopwords if len(word) > 2 or word in important_symbols]
    print(f"6. After length filter (>2): {words_after_length}")
    
    result = ' '.join(words_after_length)
    print(f"7. Final result: '{result}'")
    print("-" * 50)
    return result

# Testar com exemplos problem?ticos
test_cases = [
    "Get $500 bonus!",
    "Only $19.99 today!",
    "Win ?1000 prize!",
    "Cost ?250 only"
]

for test in test_cases:
    debug_normalize_text(test)

In [ ]:
# Testar com exemplos que cont?m s?mbolos matem?ticos e financeiros
math_test_cases = [
    "Get $500 + 20% bonus = $600 total!",
    "Discount: 50% - 10% = 40% off today!",
    "Calculate: 2 + 2 = 4, 10 * 5 = 50",
    "Price: ?100 > ?80 (save ?20!)",
    "Rate: 5% < 10% but > 2%",
    "Win ?250 ? 2 = ?500 prize!",
    "Cost: $19.99 ? 2 = $9.99 each"
]

print("=== TESTE COM S?MBOLOS MATEM?TICOS E FINANCEIROS ===")
for i, text in enumerate(math_test_cases, 1):
    normalized = normalize_text(text)
    print(f"\n{i}. Original: {text}")
    print(f"   Normalizado: {normalized}")
    
    # Verificar se s?mbolos importantes foram preservados
    important_symbols_in_text = {
        'financial': ['$', '?', '?', '?', '?', '?'],
        'mathematical': ['+', '-', '*', '/', '=', '<', '>', '?', '?', '?', '?', '?', '?'],
        'other': ['%', '#', '@', '&', '!', '?']
    }
    
    found_symbols = []
    for category, symbols in important_symbols_in_text.items():
        for sym in symbols:
            if sym in text and sym in normalized:
                found_symbols.append(f"{sym}({category})")
            elif sym in text and sym not in normalized:
                found_symbols.append(f"?{sym}({category})")
    
    if found_symbols:
        print(f"   S?mbolos preservados: {', '.join(found_symbols)}")

In [ ]:
# Apply the normalize_text function to the text column
df['normalized_text'] = df['text'].apply(normalize_text)

# Display the first few rows to see the normalized text
print(df[['text', 'normalized_text']].head())

### 2. Extra??o de features



In [ ]:
# Apply each rule function from the ruleset module to create new columns
df['rule1'] = df['normalized_text'].apply(ruleset.rule1)
df['rule2'] = df['normalized_text'].apply(ruleset.rule2)
df['rule3'] = df['normalized_text'].apply(ruleset.rule3)
df['rule4'] = df['normalized_text'].apply(ruleset.rule4)
df['rule5'] = df['normalized_text'].apply(ruleset.rule5)
df['rule6'] = df['normalized_text'].apply(ruleset.rule6)
df['rule7'] = df['normalized_text'].apply(ruleset.rule7)
df['rule8'] = df['normalized_text'].apply(ruleset.rule8)
df['rule9'] = df['normalized_text'].apply(ruleset.rule9)

# Display the dataframe with all rule columns
print("Shape after adding rule columns:", df.shape)
df.head()

In [ ]:
import os
# Remove the problematic environment variable
if 'MPLBACKEND' in os.environ:
    del os.environ['MPLBACKEND']

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Convert labels to binary values (ham=0, spam=1)
df['binary_label'] = df['label'].map({'ham': 0, 'spam': 1})

# Extract features (all rule columns) and target variable
X = df[['rule1', 'rule2', 'rule3', 'rule4', 'rule5', 'rule6', 'rule7', 'rule8', 'rule9']]
y = df['binary_label']

# Split the data into training and testing sets (80% training, 20% testing)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Create and train a Decision Tree Classifier
dt_classifier = DecisionTreeClassifier(random_state=42)
dt_classifier.fit(X_train, y_train)

# Make predictions on test data
y_pred = dt_classifier.predict(X_test)

# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
class_report = classification_report(y_test, y_pred)
conf_matrix = confusion_matrix(y_test, y_pred)

# Print the results
print(f"Decision Tree Classifier Results:")
print(f"Accuracy: {accuracy * 100:.2f}%")
print("\nClassification Report:")
print(class_report)
print("\nConfusion Matrix:")
print(conf_matrix)

# Calculate feature importances
feature_importances = dt_classifier.feature_importances_
feature_names = X.columns
importance_df = pandas.DataFrame({'Feature': feature_names, 'Importance': feature_importances})
importance_df = importance_df.sort_values('Importance', ascending=False)

print("\nFeature Importance:")
print(importance_df)

# Print true negative rate as mentioned in the paper
tn, fp, fn, tp = conf_matrix.ravel()
tnr = tn / (tn + fp)
print(f"\nTrue Negative Rate: {tnr * 100:.2f}%")
print(f"True Positive Rate (Sensitivity/Recall): {tp / (tp + fn) * 100:.2f}%")
print(f"False Positive Rate: {fp / (fp + tn) * 100:.2f}%")
print(f"False Negative Rate: {fn / (fn + tp) * 100:.2f}%")

In [ ]:
# Make predictions on test data
y_pred = dt_classifier.predict(X_train)

# Evaluate the model
accuracy = accuracy_score(y_train, y_pred)
class_report = classification_report(y_train, y_pred)
conf_matrix = confusion_matrix(y_train, y_pred)

# Print the results
print(f"Decision Tree Classifier Results:")
print(f"Accuracy: {accuracy * 100:.2f}%")
print("\nClassification Report:")
print(class_report)
print("\nConfusion Matrix:")
print(conf_matrix)

# Calculate feature importances
feature_importances = dt_classifier.feature_importances_
feature_names = X.columns
importance_df = pandas.DataFrame({'Feature': feature_names, 'Importance': feature_importances})
importance_df = importance_df.sort_values('Importance', ascending=False)

print("\nFeature Importance:")
print(importance_df)

# Print true negative rate as mentioned in the paper
tn, fp, fn, tp = conf_matrix.ravel()
tnr = tn / (tn + fp)
print(f"\nTrue Negative Rate: {tnr * 100:.2f}%")
print(f"True Positive Rate (Sensitivity/Recall): {tp / (tp + fn) * 100:.2f}%")
print(f"False Positive Rate: {fp / (fp + tn) * 100:.2f}%")
print(f"False Negative Rate: {fn / (fn + tp) * 100:.2f}%")

In [ ]:
## An?lise de Overfitting

# Vamos usar valida??o cruzada para verificar se h? overfitting
from sklearn.model_selection import cross_val_score, validation_curve, learning_curve
import numpy as np

# 1. Valida??o Cruzada com 5 folds
print("=== AN?LISE DE OVERFITTING ===\n")

# Valida??o cruzada
cv_scores = cross_val_score(dt_classifier, X, y, cv=5, scoring='accuracy')
print(f"Valida??o Cruzada (5-fold):")
print(f"Scores: {cv_scores}")
print(f"M?dia: {cv_scores.mean():.4f} (+/- {cv_scores.std() * 2:.4f})")
print(f"Desvio padr?o: {cv_scores.std():.4f}")

# Se o desvio padr?o for muito alto, pode indicar overfitting
if cv_scores.std() > 0.02:
    print("??  ALERTA: Alto desvio padr?o pode indicar overfitting!")
else:
    print("? Desvio padr?o baixo - modelo parece est?vel")

print("\n" + "="*50)

In [ ]:
# 2. An?lise da Distribui??o das Features
print("\n=== AN?LISE DAS FEATURES ===")

# Verificar distribui??o das regras
feature_distribution = df[['rule1', 'rule2', 'rule3', 'rule4', 'rule5', 'rule6', 'rule7', 'rule8', 'rule9']].sum()
total_samples = len(df)

print("Distribui??o das regras (quantos SMS triggeram cada regra):")
for rule, count in feature_distribution.items():
    percentage = (count / total_samples) * 100
    print(f"{rule}: {count}/{total_samples} ({percentage:.1f}%)")

# Verificar quantos SMS n?o triggeraram nenhuma regra
no_rules_triggered = df[(df[['rule1', 'rule2', 'rule3', 'rule4', 'rule5', 'rule6', 'rule7', 'rule8', 'rule9']].sum(axis=1) == 0)]
print(f"\nSMS que n?o triggeram NENHUMA regra: {len(no_rules_triggered)}")
print(f"Destes, quantos s?o spam: {len(no_rules_triggered[no_rules_triggered['label'] == 'spam'])}")
print(f"Destes, quantos s?o ham: {len(no_rules_triggered[no_rules_triggered['label'] == 'ham'])}")

# Verificar correla??o entre regras
correlation_matrix = df[['rule1', 'rule2', 'rule3', 'rule4', 'rule5', 'rule6', 'rule7', 'rule8', 'rule9']].corr()
print(f"\nCorrela??es mais altas entre regras:")
for i in range(len(correlation_matrix.columns)):
    for j in range(i+1, len(correlation_matrix.columns)):
        corr_value = correlation_matrix.iloc[i, j]
        if abs(corr_value) > 0.3:  # Correla??o moderada ou alta
            print(f"{correlation_matrix.columns[i]} vs {correlation_matrix.columns[j]}: {corr_value:.3f}")

print("\n" + "="*50)

In [ ]:
# 3. An?lise espec?fica da Rule4 (dominante)
print("\n=== AN?LISE DETALHADA DA RULE4 ===")

# Analisar a performance da rule4 sozinha
rule4_analysis = df.groupby(['rule4', 'label']).size().unstack(fill_value=0)
print("Distribui??o Rule4 vs Label:")
print(rule4_analysis)

# Calcular m?tricas se us?ssemos apenas a rule4
rule4_only_accuracy = ((rule4_analysis.loc[0, 'ham'] + rule4_analysis.loc[1, 'spam']) / len(df))
print(f"\nSe us?ssemos APENAS a Rule4:")
print(f"Acur?cia: {rule4_only_accuracy:.4f} ({rule4_only_accuracy*100:.2f}%)")

# Verificar quantos spam/ham triggeraram rule4
spam_with_rule4 = len(df[(df['label'] == 'spam') & (df['rule4'] == 1)])
total_spam = len(df[df['label'] == 'spam'])
ham_with_rule4 = len(df[(df['label'] == 'ham') & (df['rule4'] == 1)])
total_ham = len(df[df['label'] == 'ham'])

print(f"\nRule4 detecta {spam_with_rule4}/{total_spam} spam ({spam_with_rule4/total_spam*100:.1f}%)")
print(f"Rule4 ? triggerrada por {ham_with_rule4}/{total_ham} ham ({ham_with_rule4/total_ham*100:.1f}%)")

# Isso mostra se a rule4 ? muito espec?fica
if ham_with_rule4/total_ham < 0.02:  # Menos de 2% dos ham triggeram rule4
    print("??  Rule4 pode estar sendo muito espec?fica para spam!")

print("\n" + "="*50)

In [ ]:
# 4. Teste sem a Rule4 dominante
print("\n=== TESTE SEM RULE4 ===")

# Treinar modelo sem a rule4
X_without_rule4 = df[['rule1', 'rule2', 'rule3', 'rule5', 'rule6', 'rule7', 'rule8', 'rule9']]
y = df['binary_label']

# Split dos dados
X_train_no4, X_test_no4, y_train_no4, y_test_no4 = train_test_split(
    X_without_rule4, y, test_size=0.2, random_state=42
)

# Treinar novo modelo
dt_no_rule4 = DecisionTreeClassifier(random_state=42)
dt_no_rule4.fit(X_train_no4, y_train_no4)

# Fazer predi??es
y_pred_no4 = dt_no_rule4.predict(X_test_no4)
accuracy_no4 = accuracy_score(y_test_no4, y_pred_no4)

print(f"Acur?cia SEM Rule4: {accuracy_no4:.4f} ({accuracy_no4*100:.2f}%)")
print(f"Queda de performance: {(accuracy - accuracy_no4)*100:.2f} pontos percentuais")

# Feature importance sem rule4
feature_imp_no4 = dt_no_rule4.feature_importances_
feature_names_no4 = X_without_rule4.columns
importance_df_no4 = pandas.DataFrame({'Feature': feature_names_no4, 'Importance': feature_imp_no4})
importance_df_no4 = importance_df_no4.sort_values('Importance', ascending=False)

print(f"\nNova distribui??o de import?ncia (sem rule4):")
for idx, row in importance_df_no4.iterrows():
    print(f"{row['Feature']}: {row['Importance']:.3f}")

# Valida??o cruzada sem rule4
cv_scores_no4 = cross_val_score(dt_no_rule4, X_without_rule4, y, cv=5, scoring='accuracy')
print(f"\nValida??o cruzada sem rule4: {cv_scores_no4.mean():.4f} (+/- {cv_scores_no4.std() * 2:.4f})")

print("\n" + "="*50)

In [ ]:
# 5. Conclus?es sobre Overfitting
print("\n=== CONCLUS?ES SOBRE OVERFITTING ===")

print("? EVID?NCIAS CONTRA OVERFITTING:")
print("? Valida??o cruzada est?vel (std = 0.32%)")
print("? Modelo sem rule4 ainda tem 93.54% de acur?cia")
print("? 2516 SMS n?o triggeram nenhuma regra, sendo 2502 ham e apenas 14 spam")

print("\n??  POSS?VEIS PREOCUPA??ES:")
print("? Rule4 domina com 90.2% de import?ncia")
print("? Rule4 sozinha j? d? 97.24% de acur?cia")
print("? Rule4 triggera apenas 1.2% dos ham vs 87.3% dos spam")
print("? Rule9 (email) nunca ? triggerrada (0% dos casos)")

print("\n?? INTERPRETA??O:")
print("? N?o h? overfitting cl?ssico (modelo generaliza bem)")
print("? Mas h? DEPEND?NCIA EXCESSIVA da rule4 (n?meros de telefone)")
print("? Isso pode ser um problema de generaliza??o para outros tipos de spam")
print("? Rule4 pode estar sendo um 'atalho' muito espec?fico para este dataset")

print(f"\n?? RESUMO NUM?RICO:")
print(f"? Acur?cia original: {accuracy*100:.2f}%")
print(f"? Acur?cia sem rule4: {accuracy_no4*100:.2f}%")
print(f"? Contribui??o da rule4: {(accuracy - accuracy_no4)*100:.2f} pontos percentuais")
print(f"? Rule4 = {(accuracy - accuracy_no4)/accuracy*100:.1f}% da performance total")

print("\n" + "="*60)